In [1]:
import numpy as np

In [2]:
class Layer:
  def __init__(self):
    self.input = None
    self.output = None
  # Computes the output Z of a layer for the given input x
  def forward_propagation(self, input):
    raise NotImplementedError

  def backward_propagation(self, output_error, learning_rate):
    raise NotImplementedError

In [3]:
class FCLayer(Layer):
  # input_size = number of input neurons
  # output_size = number of output neurons

  def __init__(self, input_size, output_size):
    self.weights = np.random.rand(input_size, output_size) - 0.5
    self.bias = np.random.rand(1, output_size) - 0.5

  def forward_propagation(self, input_data):
    self.input = input_data
    self.output = np.dot(self.input, self.weights) + self.bias
    return self.output

  # Computes dL/dW, dL/dB for a given output_error = dL/dZ. Returns input_error = dL/dX which is than used as an output_error for next (previous) layer
  def backward_propagation(self, output_error, learning_rate):
    input_error = np.dot(output_error, self.weights.T)
    weights_error = np.dot(self.input.T, output_error)
    bias_error = output_error

    self.weights -= learning_rate * weights_error
    self.bias -= learning_rate * bias_error

    return input_error

In [4]:
class ActivationLayer(Layer):

  def __init__(self, activation, activation_prime):
    self.activation = activation
    self.activation_prime = activation_prime

  def forward_propagation(self, input_data):
      self.input = input_data
      self.output = self.activation(self.input)
      return self.output

  # Learning rate not used because there is no learnable parameters
  def backward_propagation(self, output_error, learning_rate):
    return self.activation_prime(self.input) * output_error

In [5]:
# activation fn and its derivative
def tanh(x):
    return np.tanh(x);


def tanh_prime(x):
    return 1-np.tanh(x)**2;


def sigmoid(x):
  return 1 / (1 + np.exp(-x))


def sigmoid_prime(x):
  return sigmoid(x)*(1 - sigmoid(x))


def mse(y_true, y_pred):
  return np.mean(np.power(y_true-y_pred, 2));


def mse_prime(y_true, y_pred):
  return 2*(y_pred-y_true)/y_true.size;

In [6]:
class Network:
  def __init__(self):
    self.layers = []
    self.loss = None
    self.loss_prime = None

  def add(self, layer):
      self.layers.append(layer)

  def use(self, loss, loss_prime):
    self.loss = loss
    self.loss_prime = loss_prime

  def predict(self, input_data):

    samples = len(input_data)
    result = []

    for i in range(samples):
      # Forward propagation
      output = input_data[i]

      for layer in self.layers:
          output = layer.forward_propagation(output)
      result.append(output)

    return result

  # Train the neural net
  def fit(self, x_train, y_train, epochs, learning_rate):

    samples = len(x_train)

    for i in range(epochs):
      err = 0
      for j in range(samples):
        # Forward propagation
          output = x_train[j]
          for layer in self.layers:
            output = layer.forward_propagation(output)
          # Compute the loss
          err += self.loss(y_train[j], output)

          # Backward propagation
          error = self.loss_prime(y_train[j], output)
          for layer in reversed(self.layers):
            error = layer.backward_propagation(error, learning_rate)
                  # calculate average error on all samples
      err /= samples
      print('epoch %d/%d   error=%f' % (i+1, epochs, err))

In [ ]:
# x_train je trodimenzionalni niz kako bi se omogućilo skaliranje na batch treniranje u budućnosti.
# Prva dimenzija predstavlja broj uzoraka, dok druga dimenzija (1) omogućava da se svaki uzorak tretira kao mala matrica.
# Iako trenutno treniramo po uzorku, struktura sa matricama omogućava konzistentno matrično množenje (np.dot)
# u slojevima mreže, čime olakšavamo proširenje na mini-batch treniranje kasnije.

x_train = np.array([[[0, 0, 1]], [[0, 1, 1]], [[1, 0, 1]], [[1, 1, 1]]])
y_train = np.array([[[0]], [[1]], [[1]], [[0]]])

# network
net = Network()
net.add(FCLayer(3, 4))
net.add(ActivationLayer(sigmoid, sigmoid_prime))
net.add(FCLayer(4, 1))
net.add(ActivationLayer(sigmoid, sigmoid_prime))

# train
net.use(mse, mse_prime)
net.fit(x_train, y_train, epochs=10000, learning_rate=0.1)

# test
out = net.predict(x_train)
print(out)

In [ ]:
# This was written when I was a complete beginer so it would be good to check if there are some mistakes